<a href="https://colab.research.google.com/github/hemachandranrcs24-spec/Ai-Lab-5H/blob/main/1bm24cs115_Hemachandranr_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 8-Puzzle Solver with Nodes Explored Tracker and Matrix Box UI
import sys

def print_matrix_box(state, label=""):
    """Prints a single 9-element state configuration in a clean, visual box grid."""
    if label:
        print(f"--- {label} ---")

    # Extract values into rows, replacing 0 with a blank space for visibility
    vals = [str(x) if x != 0 else " " for x in state]

    print("┌───┬───┬───┐")
    print(f"│ {vals[0]} │ {vals[1]} │ {vals[2]} │")
    print("├───┼───┼───┤")
    print(f"│ {vals[3]} │ {vals[4]} │ {vals[5]} │")
    print("├───┼───┼───┤")
    print(f"│ {vals[6]} │ {vals[7]} │ {vals[8]} │")
    print("└───┴───┴───┘")

def get_moves(state):
    """Finds valid next states from the current state (0 represents the blank space)."""
    moves = []
    blank_idx = state.index(0)
    row, col = blank_idx // 3, blank_idx % 3

    # Standard tile sliding directions (Up, Down, Left, Right)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for dr, dc in directions:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col
            new_state = list(state)
            new_state[blank_idx], new_state[new_idx] = new_state[new_idx], new_state[blank_idx]
            moves.append(tuple(new_state))
    return moves

# ==========================================
# 1. Depth First Search (DFS) Implementation
# ==========================================
def solve_dfs(initial, goal, max_depth=25):
    stack = [(initial, [initial])]
    visited = set()
    nodes_explored = 0

    while stack:
        state, path = stack.pop()

        if state not in visited:
            visited.add(state)
            nodes_explored += 1

            if state == goal:
                return path, nodes_explored

            if len(path) - 1 >= max_depth:
                continue

            for next_state in get_moves(state):
                if next_state not in visited:
                    stack.append((next_state, path + [next_state]))
    return None, nodes_explored

# ====================================================
# 2. Iterative Deepening Search (IDS) Implementation
# ====================================================
def depth_limited_search(state, goal, depth, visited, path, tracking_dict):
    tracking_dict['nodes_explored'] += 1

    if state == goal:
        return path
    if depth <= 0:
        return None

    visited.add(state)
    for next_state in get_moves(state):
        if next_state not in visited:
            result = depth_limited_search(next_state, goal, depth - 1, visited, path + [next_state], tracking_dict)
            if result is not None:
                return result

    visited.remove(state)  # Backtracking clean up
    return None

def solve_ids(initial, goal, max_depth=50):
    total_nodes_explored = 0
    for depth in range(max_depth):
        visited = set()
        tracking_dict = {'nodes_explored': 0}
        path = depth_limited_search(initial, goal, depth, visited, [initial], tracking_dict)
        total_nodes_explored += tracking_dict['nodes_explored']
        if path:
            return path, total_nodes_explored
    return None, total_nodes_explored

def parse_input(prompt_text):
    print(prompt_text)
    print("Enter 9 numbers space-separated or line by line (use 0 for blank space):")
    inputs = []
    while len(inputs) < 9:
        line = input().strip()
        if not line:
            continue
        inputs.extend([int(x) for x in line.split() if x.isdigit() or x == '0'])
    return tuple(inputs[:9])

if __name__ == "__main__":
    print("--- 8-Puzzle Solver Configuration ---")
    print("Example layout format input: 5 4 6 1 8 7 3 2 0\n")

    try:
        initial = parse_input(">> Enter INITIAL STATE:")
        goal = parse_input(">> Enter GOAL STATE:")
    except ValueError:
        print("Invalid numeric configuration format. Exiting.")
        sys.exit(1)

    print("\nInitial Configuration Entered:")
    print_matrix_box(initial)

    print("Target Goal Configuration Entered:")
    print_matrix_box(goal)

    print("\n--- Executing IDS ---")
    ids_path, ids_nodes = solve_ids(initial, goal, max_depth=30)
    if ids_path:
        print(f"SUCCESS: Goal reached in {len(ids_path)-1} moves.")
        print(f"Total nodes/states explored during IDS: {ids_nodes}\n")

        show_path = input("Would you like to print out the complete matrix box transitions step-by-step? (y/n): ").strip().lower()
        if show_path == 'y':
            for step, state in enumerate(ids_path):
                print_matrix_box(state, label=f"Step {step}")
    else:
        print("IDS could not find a solution within depth limits.")

    print("\n--- Executing DFS (Bounded to Depth 20 to safeguard memory) ---")
    dfs_path, dfs_nodes = solve_dfs(initial, goal, max_depth=20)
    if dfs_path:
        print(f"SUCCESS: Goal reached via bounded DFS!")
        print(f"Total nodes/states explored during DFS: {dfs_nodes}")
    else:
        print(f"DFS did not locate the target within depth bounds. States explored: {dfs_nodes}")


--- 8-Puzzle Solver Configuration ---
Example layout format input: 5 4 6 1 8 7 3 2 0

>> Enter INITIAL STATE:
Enter 9 numbers space-separated or line by line (use 0 for blank space):
2
8
3
1
6
4
7
0
5
>> Enter GOAL STATE:
Enter 9 numbers space-separated or line by line (use 0 for blank space):
1
2
3
8
0
4
7
6
5

Initial Configuration Entered:
┌───┬───┬───┐
│ 2 │ 8 │ 3 │
├───┼───┼───┤
│ 1 │ 6 │ 4 │
├───┼───┼───┤
│ 7 │   │ 5 │
└───┴───┴───┘
Target Goal Configuration Entered:
┌───┬───┬───┐
│ 1 │ 2 │ 3 │
├───┼───┼───┤
│ 8 │   │ 4 │
├───┼───┼───┤
│ 7 │ 6 │ 5 │
└───┴───┴───┘

--- Executing IDS ---
SUCCESS: Goal reached in 5 moves.
Total nodes/states explored during IDS: 73

Would you like to print out the complete matrix box transitions step-by-step? (y/n): y
--- Step 0 ---
┌───┬───┬───┐
│ 2 │ 8 │ 3 │
├───┼───┼───┤
│ 1 │ 6 │ 4 │
├───┼───┼───┤
│ 7 │   │ 5 │
└───┴───┴───┘
--- Step 1 ---
┌───┬───┬───┐
│ 2 │ 8 │ 3 │
├───┼───┼───┤
│ 1 │   │ 4 │
├───┼───┼───┤
│ 7 │ 6 │ 5 │
└───┴───┴───┘
--- Step 2